# Compare CellViT Inference Runs (Tables + Disagreement Images)

This notebook:
1. Defines a `MODEL_RUNS` dict mapping model labels → run directories.
2. Runs inference for each run directory (optional / can skip if results already exist).
3. Loads each `inference_results.json` and builds:
   - Overall metrics table
   - Per-class PQ table
   - Per-class F1/Precision/Recall table
4. Computes "most different" images between two models (by `bPQ` or `Dice` deltas).

> Tip: run the **inference** step only when your training jobs are finished (or point to already-finished run dirs).


In [2]:
from __future__ import annotations

import json
import math
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [3]:
# ----------------------------
# 1) Configure paths + runs
# ----------------------------

# Set this to the AI-GUIDED-CLEAN root on SCC
ROOT_PATH = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN")

INFER_SCRIPT = ROOT_PATH / "CellViT-plus-plus/cellvit/training/evaluate/inference_cellvit_experiment_pannuke.py"

# Edit this dict for your current experiments
MODEL_RUNS: Dict[str, Path] = {
    # "SAM-H Baseline (Prev)": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local/2025-12-05T174106_tcga_finetune_256",
    "SAM-H (New)": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256",
    "FiLM": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-10T225437_First_film try same with patch size 256",
    "Virchow": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_virchow/2026-02-10T225413_VirchowTrainTestValSplit",
    "FiLM Rosie Weights": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-10T231234_FilmRosieWeights",
}

GPU_ID = 0

In [4]:
# ----------------------------
# 2) Helpers: run inference
# ----------------------------

def run_inference_for_run(run_dir: Path, gpu: int = 0, force: bool = False) -> Path:
    """Run inference script for a single run directory.
    Returns path to inference_results.json.
    """
    run_dir = Path(run_dir)
    out_json = run_dir / "inference_results.json"

    if out_json.exists() and not force:
        print(f"✅ exists, skipping: {out_json}")
        return out_json

    cmd = [
        "python",
        str(INFER_SCRIPT),
        "--run_dir", str(run_dir),
        "--gpu", str(gpu),
    ]

    print("▶️", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not out_json.exists():
        raise FileNotFoundError(f"Expected {out_json} but not found.")
    print(f"✅ wrote: {out_json}")
    return out_json


def run_all_inference(model_runs: Dict[str, Path], gpu: int = 0, force: bool = False) -> Dict[str, Path]:
    results = {}
    for name, rd in model_runs.items():
        print(f"\n=== {name} ===")
        results[name] = run_inference_for_run(rd, gpu=gpu, force=force)
    return results

In [5]:
# ----------------------------
# 3) Helpers: parse json → tables
# ----------------------------

def _safe_float(x):
    try:
        if x is None:
            return float("nan")
        if isinstance(x, str):
            # handle "nan"
            if x.lower() == "nan":
                return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def load_inference_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def build_overall_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        ds = d.get("dataset", {})
        row = {"model": model_name}
        for k, v in ds.items():
            row[k] = _safe_float(v)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model")
    # nicer ordering (keep common metrics first if present)
    preferred = ["mPQ", "mDQ", "mSQ", "bPQ", "bDQ", "bSQ", "Binary-Cell-Dice-Mean", "Binary-Cell-Jacard-Mean",
                 "f1_detection", "precision_detection", "recall_detection", "Tissue-Multiclass-Accuracy"]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    return df[cols].sort_index()


def build_perclass_pq_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        pq = d.get("nuclei_metrics_pq", {})
        row = {"model": model_name}
        for cls, val in pq.items():
            row[cls] = _safe_float(val)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").sort_index()


def build_perclass_detection_table(results: Dict[str, dict]) -> pd.DataFrame:
    # nuclei_metrics_d: {class: {f1_cell, prec_cell, rec_cell}}
    rows = []
    for model_name, d in results.items():
        det = d.get("nuclei_metrics_d", {})
        row = {"model": model_name}
        for cls, stats in det.items():
            if not isinstance(stats, dict):
                continue
            row[f"{cls}__f1"] = _safe_float(stats.get("f1_cell"))
            row[f"{cls}__prec"] = _safe_float(stats.get("prec_cell"))
            row[f"{cls}__rec"] = _safe_float(stats.get("rec_cell"))
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model").sort_index()
    return df

In [6]:
# ----------------------------
# 4) Run inference (optional)
# ----------------------------
# If your inference_results.json already exist, keep force=False.
# If you want to rerun (overwrite), set force=True.

FORCE_RERUN = False

# Uncomment to run:
# json_paths = run_all_inference(MODEL_RUNS, gpu=GPU_ID, force=FORCE_RERUN)

In [7]:
# ----------------------------
# 5) Load jsons + build tables
# ----------------------------

def load_all_results(model_runs: Dict[str, Path]) -> Dict[str, dict]:
    out = {}
    for name, rd in model_runs.items():
        p = rd / "inference_results.json"
        if not p.exists():
            raise FileNotFoundError(f"Missing {p}. Run inference first.")
        out[name] = load_inference_json(p)
    return out

results = load_all_results(MODEL_RUNS)

overall_df = build_overall_table(results)
perclass_pq_df = build_perclass_pq_table(results)
perclass_det_df = build_perclass_detection_table(results)

display(overall_df)

,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,Binary-Cell-Jacard-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
model,,,,,,,,,,,,
FiLM,0.560165,0.673290,0.695389,0.661408,0.796692,0.803676,0.776244,0.672461,0.841105,0.846675,0.835608,1.0
FiLM Rosie Weights,0.566806,0.683589,0.716408,0.653192,0.788479,0.802493,0.776458,0.670383,0.837916,0.835016,0.840836,1.0
SAM-H (New),0.547703,0.659462,0.692783,0.658654,0.795331,0.818764,0.774030,0.663925,0.828375,0.836095,0.820796,1.0
Virchow,0.542031,0.661382,0.673654,0.639662,0.781103,0.786847,0.766659,0.656326,0.832997,0.825864,0.840256,1.0


In [8]:
# Per-class PQ
display(perclass_pq_df)

# Per-class detection metrics (F1/Prec/Rec)
display(perclass_det_df)

,epithelial,lymphocyte,macrophage,neutrophil,other
model,,,,,
FiLM,0.620232,0.445084,0.235810,0.391008,NaN
FiLM Rosie Weights,0.625844,0.502487,0.209404,0.315933,NaN
SAM-H (New),0.623577,0.477741,0.159297,0.339665,NaN
Virchow,0.616424,0.402067,0.214479,0.329779,NaN


,epithelial__f1,epithelial__prec,epithelial__rec,lymphocyte__f1,lymphocyte__prec,lymphocyte__rec,macrophage__f1,macrophage__prec,macrophage__rec,neutrophil__f1,neutrophil__prec,neutrophil__rec,other__f1,other__prec,other__rec
model,,,,,,,,,,,,,,,
FiLM,0.822879,0.836047,0.810120,0.823300,0.809341,0.837748,0.406593,0.672727,0.291339,0.425532,0.333333,0.588235,NaN,NaN,NaN
FiLM Rosie Weights,0.809194,0.841590,0.779200,0.798760,0.747100,0.858095,0.411111,0.660714,0.298387,0.520548,0.475000,0.575758,NaN,NaN,NaN
SAM-H (New),0.812051,0.826674,0.797936,0.815515,0.798341,0.833444,0.278788,0.676471,0.175573,0.380000,0.287879,0.558824,NaN,NaN,NaN
Virchow,0.815074,0.803609,0.826871,0.817329,0.801014,0.834323,0.422857,0.698113,0.303279,0.542857,0.527778,0.558824,NaN,NaN,NaN


In [9]:
# Save tables to disk (optional)
OUT_DIR = Path("./comparison_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

overall_df.to_csv(OUT_DIR / "model_overall_metrics.csv")
perclass_pq_df.to_csv(OUT_DIR / "perclass_pq.csv")
perclass_det_df.to_csv(OUT_DIR / "perclass_detection_metrics.csv")

print("Wrote:", OUT_DIR)

Wrote: comparison_outputs


In [10]:
# ----------------------------
# 6) Image-level disagreement
# ----------------------------
# Each inference json has image_metrics: {image_id: {Dice, Jaccard, bPQ, ...}}
# We'll compute top-K images where modelA and modelB differ most by a chosen metric.

def image_metric_df(d: dict, metric: str = "bPQ") -> pd.DataFrame:
    im = d.get("image_metrics", {})
    rows = []
    for image_id, m in im.items():
        if isinstance(m, dict) and metric in m:
            rows.append({"image_id": image_id, metric: _safe_float(m[metric])})
    return pd.DataFrame(rows).set_index("image_id")

def top_disagreement(modelA: str, modelB: str, metric: str = "bPQ", k: int = 20) -> pd.DataFrame:
    a = image_metric_df(results[modelA], metric=metric)
    b = image_metric_df(results[modelB], metric=metric)
    df = a.join(b, lsuffix=f"__{modelA}", rsuffix=f"__{modelB}", how="inner")
    df["delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"]).abs()
    df["signed_delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"])
    df = df.sort_values("delta", ascending=False).head(k)
    return df

BASELINE_NAME = "SAM-H (New)"
COMPARE_NAME = "FiLM Rosie Weights"  # change to any model label
METRIC = "bPQ"

disagree_df = top_disagreement(BASELINE_NAME, COMPARE_NAME, metric=METRIC, k=30)
display(disagree_df)

,bPQ__SAM-H (New),bPQ__FiLM Rosie Weights,delta,signed_delta
image_id,,,,
TCGA-G9-6499-01Z-00-DX1-1_768_0.png,0.804122,0.000000,0.804122,-0.804122
TCGA-G9-6499-01Z-00-DX1-1_512_0.png,0.322637,0.000000,0.322637,-0.322637
TCGA-G9-6499-01Z-00-DX1-1_256_0.png,0.387675,0.621267,0.233592,0.233592
TCGA-KK-A59X-01Z-00-DX1-2_0_0.png,0.637948,0.866653,0.228705,0.228705
TCGA-G9-6499-01Z-00-DX1-1_512_256.png,0.219601,0.000000,0.219601,-0.219601
TCGA-E9-A22G-01Z-00-DX1_4_0_0.png,0.538138,0.388361,0.149776,-0.149776
TCGA-KK-A6E0-01Z-00-DX1-2_0_0.png,0.595654,0.470606,0.125048,-0.125048
TCGA-EW-A6SD-01Z-00-DX1_4_256_0.png,0.583214,0.475859,0.107355,-0.107355
TCGA-EW-A6SD-01Z-00-DX1_3_0_0.png,0.596167,0.489324,0.106843,-0.106843


## Next step (visualization)
You said you already have a notebook that can visualize a specific image_id across:
- GT
- prediction A
- prediction B

Once you confirm the disagreement table looks good, we can:
1. Take the top-K `image_id`s (best/worst or highest delta),
2. Feed them into your visualization helper to render side-by-side.
